In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict
from rich import print
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END

In [2]:
load_dotenv(override=True)
MODEL_NAME = "gemini-3.8-flash"
MODEL_PROVIDER = "google_genai"

In [3]:
class State(TypedDict):
    question: str
    answer: str

In [4]:
llm = init_chat_model(model=MODEL_NAME, model_provider=MODEL_PROVIDER)

In [5]:
def answer_node(State: State) -> dict:
    responce = llm.invoke(State["question"])
    
    # Extract clean text if Gemini returns list of chunks/signatures
    if isinstance(responce.content, list):
        clean_text = " ".join([
            item["text"] for item in responce.content 
            if isinstance(item, dict) and "text" in item
        ]).strip()
    else:
        clean_text = str(responce.content)
        
    return {"answer": clean_text}

In [6]:
def build_graph():
    graph = StateGraph(State)

    graph.add_node("answer", answer_node)
    graph.add_edge(START, "answer")
    graph.add_edge("answer", END)

    return graph.compile()

In [7]:
agent = build_graph()

In [8]:
agent

In [9]:
agent_result = agent.invoke({"question": "what is python explain in single sentence"})
print(f"[bold cyan]Que:[/bold cyan] {agent_result['question']}")
print(f"[bold magenta]Ans:[/bold magenta] {agent_result['answer']}")